# 04 CNN ????

## ????

- ??????? `[N, C, H, W]` ???
- ????????????????
- ??????????? CNN?
- ?? loss ??????????????

## ????

?????????????????????????

CNN ??????????????????????????????????????????????????????

## ????

- Channel?????????? 1?RGB ?? 3?
- Convolution??????????????
- Pooling????????????????
- Flatten????????????????????

## ??????

??????????????????? shape??????????? `[N, H, W]` ? `[N, C, H, W]`?

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

digits = load_digits()
images = digits.images.astype(np.float32) / 16.0
labels = digits.target.astype(np.int64)

x_train, x_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

x_train = torch.tensor(x_train).unsqueeze(1)
x_test = torch.tensor(x_test).unsqueeze(1)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print("train image shape:", x_train.shape)

plt.figure(figsize=(6, 2))
for i in range(6):
    plt.subplot(1, 6, i + 1)
    plt.imshow(x_train[i, 0], cmap="gray")
    plt.title(str(y_train[i].item()))
    plt.axis("off")
plt.tight_layout()
plt.show()

## ????

???????????? CNN????????????????????????????????

### ??????

???????? CNN???? loss ???????????????????????? shape???????? `in_features`?

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)
model = TinyCNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []

for epoch in range(20):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
    losses.append(total_loss / len(train_loader.dataset))

model.eval()
with torch.no_grad():
    test_logits = model(x_test)
    test_pred = test_logits.argmax(dim=1)
    test_acc = (test_pred == y_test).float().mean().item()

print(f"final loss={losses[-1]:.4f}, test accuracy={test_acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("CNN training loss")
plt.xlabel("Epoch")
plt.ylabel("Cross entropy")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 2.5))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(x_test[i, 0], cmap="gray")
    plt.title(f"p:{test_pred[i].item()}\ny:{y_test[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## ????

??????????loss ???????????????????????????????????? CNN ?????????????????????????????

## ????

- ???? channel ??????? `[N, H, W]`?
- `nn.Linear` ???????????
- ???? `torch.long`?
- ???????? `model.eval()` ? `torch.no_grad()`?

??????????????????????????????????????? shape?

In [ ]:
sample = x_train[:4]
with torch.no_grad():
    feature = model.features(sample)
print("input shape:", sample.shape)
print("feature shape before classifier:", feature.shape)

## ????

**Q???? CNN ?????? flatten?**  
A??? flatten ?????????????????????????

**Q???????????**  
A?????????????????????????????????

**Q???????? channel ??**  
A??????????????????????????????? `C=1`?

## ??

CNN ???????????????????? CNN ?????????????????????????